In [27]:
import pandas as pd 

MainAtApogee_df = pd.read_csv('/Users/ricechessmaster/Documents/Dispersion Analysis v2/backend/data/Polaris_Cycle1_Aug2025-MainAtApogee.csv')
MainAtApogee_df = MainAtApogee_df.set_index('Simulation')

DrogueOnly_df = pd.read_csv('/Users/ricechessmaster/Documents/Dispersion Analysis v2/backend/data/Polaris_Cycle1_Aug2025-DrogueOnly.csv')
DrogueOnly_df = DrogueOnly_df.set_index('Simulation')

Ballistic_df = pd.read_csv('/Users/ricechessmaster/Documents/Dispersion Analysis v2/backend/data/Polaris_Cycle1_Aug2025-Ballistic.csv')
Ballistic_df = Ballistic_df.set_index('Simulation')

In [28]:
SimParams_df = pd.read_csv('/Users/ricechessmaster/Documents/Dispersion Analysis v2/backend/data/sim_parameters_historical_aug2025_filtered.csv')
altitudes = [110, 320, 500, 800, 1000, 1500, 1900, 3200, 4200, 5600, 7200, 9200, 10400, 11800, 13500, 15800, 17700, 19300, 22000]
SimParams_df = SimParams_df.rename(columns={f"direction [{alt}]": f"direction_{alt}" for alt in altitudes})
SimParams_df = SimParams_df.drop(columns=[f"stdev [{alt}]" for alt in altitudes])


In [29]:
# Order of when to join would matter eventually if there is too much data; could apply where first, then join
# left_index = True is used as the name "Simulations" for the 0th column is not real... it's a default

Full_MainAtApogee_df = pd.merge(MainAtApogee_df, SimParams_df, left_index=True, right_on='date', how="inner")
Full_DrogueOnly_df = pd.merge(DrogueOnly_df, SimParams_df, left_index=True, right_on='date', how="inner")
Full_Ballistic_df = pd.merge(Ballistic_df, SimParams_df, left_index=True, right_on='date', how="inner")


In [30]:
import numpy as np


def haversine_nm(ref_latitude:float, ref_longitude:float, latitude, longitude):
    """
    Compute the great-circle distance between two points on Earth (in nautical miles).
    """

    # Earth's mean radius in nautical miles
    R_nm = 3443.92

    # Note: input of 2 scalars and 2 arrays to be converted to radians
    ref_latitude, ref_longitude, latitude, longitude = map(np.radians, [ref_latitude, ref_longitude, latitude, longitude])

    dlat = latitude - ref_latitude
    dlon = longitude - ref_longitude

    a = np.sin(dlat / 2) ** 2 + np.cos(ref_latitude) * np.cos(latitude) * np.sin(dlon / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    return R_nm * c


# reference launch coordinates (degrees)
ref_lat = 47.965378
ref_long = -81.873536

Main_landing_latitudes = Full_MainAtApogee_df["Polaris Rocket Landing Latitude (°N)"].to_numpy()
Main_landing_longitudes = Full_MainAtApogee_df["Polaris Rocket Landing Longitude (°E)"].to_numpy()

# compute great-circle distance in nautical miles using the shared haversine helper
landing_distances = haversine_nm(ref_lat, ref_long, Main_landing_latitudes, Main_landing_longitudes)

Full_MainAtApogee_df['distance_nm'] = landing_distances
Main_safety_column = np.where(landing_distances < 10, True, False)
Full_MainAtApogee_df["safety_column"] = Main_safety_column
# print(Full_MainAtApogee_df['distance_nm'].describe())

In [31]:
DrogueOnly_landing_latitudes = Full_DrogueOnly_df["Polaris Rocket Landing Latitude (°N)"].to_numpy()
DrogueOnly_landing_longitudes = Full_DrogueOnly_df["Polaris Rocket Landing Longitude (°E)"].to_numpy()

# compute great-circle distance in nautical miles using the shared haversine helper
landing_distances = haversine_nm(ref_lat, ref_long, DrogueOnly_landing_latitudes, DrogueOnly_landing_longitudes)

# insert distances and safety column into dataframe
Full_DrogueOnly_df['distance_nm'] = landing_distances
DrogueOnly_safety_column = np.where(landing_distances < 10, True, False)
Full_DrogueOnly_df["safety_column"] = DrogueOnly_safety_column

In [32]:
# Extract and use the haversine function to compute distance from the starting point
Ballistic_landing_latitudes = Full_Ballistic_df["Polaris Rocket Landing Latitude (°N)"].to_numpy()
Ballistic_landing_longitudes = Full_Ballistic_df["Polaris Rocket Landing Longitude (°E)"].to_numpy()

# compute great-circle distance in nautical miles using the shared haversine helper
landing_distances = haversine_nm(ref_lat, ref_long, Ballistic_landing_latitudes, Ballistic_landing_longitudes)

# insert distances and safety column into dataframe
Full_Ballistic_df['distance_nm'] = landing_distances
Ballistic_safety_column = np.where(landing_distances < 10, True, False)
Full_Ballistic_df["safety_column"] = Ballistic_safety_column

In [33]:
Full_safety_col_arry = Full_MainAtApogee_df["safety_column"].to_numpy()
DrogueOnly_safety_col_arry = Full_DrogueOnly_df["safety_column"].to_numpy()
Ballistic_safety_col_arry = Full_Ballistic_df["safety_column"].to_numpy()

# Default axis to None, which would flatten the array either way
safe, unsafe  = np.count_nonzero(Full_safety_col_arry == True), np.count_nonzero(Full_safety_col_arry == False)
print(safe, unsafe)

safe, unsafe  = np.count_nonzero(DrogueOnly_safety_col_arry == True), np.count_nonzero(DrogueOnly_safety_col_arry == False)
print(safe, unsafe)

safe, unsafe  = np.count_nonzero(Ballistic_safety_col_arry == True), np.count_nonzero(Ballistic_safety_col_arry == False)
print(safe, unsafe)


227 1
228 0
228 0


In [34]:
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

column_names = ["temperature", "pressure", "Max Windspeed (mph)"]
altitudes = [110, 320, 500, 800, 1000, 1500, 1900, 3200, 4200, 5600, 7200, 9200, 10400, 11800, 13500, 15800, 17700, 19300, 22000]

for alt in altitudes:
    column_names.append(str(alt)) 
    column_names.append(f"direction_{str(alt)}") 

# Define the input and the output 
X = Full_MainAtApogee_df[column_names]
y = Full_MainAtApogee_df['distance_nm']

# 80% for training, set seed 42; stratify not necessary
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Create the empty brains
model_main_distance = XGBRegressor()
# Train it!
model_main_distance.fit(X_train, y_train)

y_model_main_distance_predictions = model_main_distance.predict(X_test)
print(predictions)

#landing_lat: Mapped[float] = mapped_column(nullable=False)
#landing_lon: Mapped[float] = mapped_column(nullable=False)
#distance_nm: Mapped[float] = mapped_column(nullable=False)
# main_model = XGBRegressor()
# train_test_split()

[5.871361  4.032478  6.739104  6.1828313 5.071712  3.944373  4.958502
 3.5302365 7.6215024 3.1949391 6.3271546 4.5190873 7.9194365 8.440969
 6.4546113 7.419286  4.3773956 4.6162767 6.308994  6.3978696 7.191475
 6.3785157 6.686252  8.009919  2.877791  4.3542485 4.7190266 6.927223
 7.465635  6.1137476 3.5349944 1.2717879 4.1470623 3.1532357 4.9761143
 3.259412  5.904449  2.663137  5.97267   7.33575   7.603537  7.859548
 7.1561217 2.4352067 5.6813307 5.493844 ]


In [35]:
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

model_main_RMS = np.sqrt(mean_squared_error(y_test, y_model_main_distance_predictions))
print(f"RMS Error: {model_main_RMS}")
score = r2_score(y_test, y_model_main_distance_predictions)
print(f"R-squared score: {score}")


RMS Error: 0.5941581098089428
R-squared score: 0.8967763807664032
